In [ ]:
import datetime
import math
import time
import threading
import rclpy

from IPython.display import clear_output
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.executors import MultiThreadedExecutor
from geographic_msgs.msg import GeoPoint
from lotusim_msgs.msg import VesselPositionArray
from lotusim_msgs.msg import MASCmd as MASCmdMsg
from lotusim_msgs.action import MASCmd, MASCmdArray

This notebook sets up a radar detection test scene:
- A static **Commando** ship at the center
- A single **LRAUV** radar target slightly offset to the front
- A ring of **30 static LRAUVs** arranged in a circle around the center

All spawn methods return an action future. Since the executor runs in a background thread, futures are polled with `while not future.done()` instead of `rclpy.spin_until_future_complete`.

In [ ]:
SPAWN_LATITUDE = 1.2605794416293148
SPAWN_LONGITUDE = 103.7516212463379
SPAWN_ALTITUDE = 0.0

class RadarTestSpawner(Node):
    def __init__(self):
        super().__init__('radar_test_spawner')

        self.pose_subscription = self.create_subscription(
            VesselPositionArray,
            '/lotusim/poses',
            self.poses_callback,
            10)
        self.mas_action_client = ActionClient(
            self,
            MASCmd,
            '/lotusim/mas_cmd'
        )
        self.mas_array_action_client = ActionClient(
            self,
            MASCmdArray,
            '/lotusim/mas_cmd_array'
        )
        self.vessel_poses = {}

    def poses_callback(self, msg):
        for vessel_position in msg.vessels:
            lat = vessel_position.geo_point.latitude
            lon = vessel_position.geo_point.longitude
            alt = vessel_position.geo_point.altitude
            self.vessel_poses[vessel_position.vessel_name] = (lat, lon, alt)

    def spawn_static_commando(self):
        """Spawn a single commando ship at the center"""
        msg = MASCmdMsg()
        msg.cmd_type = MASCmdMsg.CREATE_CMD
        msg.model_name = "commando"
        msg.vessel_name = "commando_center"

        geo = GeoPoint()
        geo.latitude = SPAWN_LATITUDE
        geo.longitude = SPAWN_LONGITUDE
        geo.altitude = SPAWN_ALTITUDE
        msg.geo_point = geo

        msg.sdf_string = """
        <lotus_param>
            <waypoint_follower>
                <follower>
                    <loop>false</loop>
                </follower>
            </waypoint_follower>
        </lotus_param>
        """
        goal_msg = MASCmd.Goal()
        goal_msg.cmd = msg
        self.mas_action_client.wait_for_server()
        return self.mas_action_client.send_goal_async(goal_msg)

    def spawn_front_target(self):
        msg = MASCmdMsg()
        msg.cmd_type = MASCmdMsg.CREATE_CMD
        msg.model_name = "lrauv"
        msg.vessel_name = "radar_target"

        geo = GeoPoint()
        # ~10 meters north of center
        geo.latitude = SPAWN_LATITUDE
        geo.longitude = SPAWN_LONGITUDE + 0.00009
        geo.altitude = SPAWN_ALTITUDE
        msg.geo_point = geo

        msg.sdf_string = """
        <lotus_param>
            <render_interface>
                <publish_render>false</publish_render>
            </render_interface>
        </lotus_param>
        """
        goal_msg = MASCmd.Goal()
        goal_msg.cmd = msg
        self.mas_action_client.wait_for_server()
        return self.mas_action_client.send_goal_async(goal_msg)

    def spawn_lrauv_circle(self, n_ships=30, radius_m=30):
        """Spawn n_ships LRAUVs in a circle around the center"""
        goal_msg = MASCmdArray.Goal()

        for i in range(n_ships):
            angle = (2 * math.pi / n_ships) * i
            lat_offset = (radius_m / 111111.0) * math.cos(angle)
            lon_offset = (radius_m / (111111.0 * math.cos(math.radians(SPAWN_LATITUDE)))) * math.sin(angle)

            msg = MASCmdMsg()
            msg.cmd_type = MASCmdMsg.CREATE_CMD
            msg.model_name = "lrauv"
            msg.vessel_name = f"lrauv_{i}"

            geo = GeoPoint()
            geo.latitude = SPAWN_LATITUDE + lat_offset
            geo.longitude = SPAWN_LONGITUDE + lon_offset
            geo.altitude = SPAWN_ALTITUDE
            msg.geo_point = geo

            msg.sdf_string = """
            <lotus_param>
                <render_interface>
                    <publish_render>false</publish_render>
                </render_interface>
            </lotus_param>
            """
            goal_msg.cmd.append(msg)

        self.mas_array_action_client.wait_for_server()
        return self.mas_array_action_client.send_goal_async(goal_msg)

In [ ]:
if not rclpy.ok():
    rclpy.init(args=None)

try:
    stop_executor()
except NameError:
    pass
except Exception as e:
    print(f"Could not destroy resources: {e}")

node = RadarTestSpawner()
executor = MultiThreadedExecutor()
executor.add_node(node)
stop_event = threading.Event()

def spin_executor():
    executor.spin()

def stop_executor():
    """Call this function to stop the spinning thread"""
    try:
        stop_event.set()
        executor.shutdown()
        spin_thread.join()
        display_thread.join()
        node.destroy_node()
        time.sleep(2)
        print("Executor stopped successfully")
    except Exception as e:
        print(f"Error stopping executor: {e}")

def display_loop():
    while not stop_event.is_set():
        clear_output(wait=True)
        print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}]")
        for name, (lat, lon, alt) in node.vessel_poses.items():
            print(f"  {name}: lat={lat:.6f}, lon={lon:.6f}, alt={alt:.2f}")
        time.sleep(1)

display_thread = threading.Thread(target=display_loop, daemon=True)
display_thread.start()
spin_thread = threading.Thread(target=spin_executor, daemon=True)
spin_thread.start()

In [ ]:
# Spawn central Commando ship
future = node.spawn_static_commando()
while not future.done():
    time.sleep(0.1)
goal_handle = future.result()
result_future = goal_handle.get_result_async()
while not result_future.done():
    time.sleep(0.1)
print(f"Commando ship spawned: {result_future.result().result.name}")

In [ ]:
# Spawn LRAUV radar target
future = node.spawn_front_target()
while not future.done():
    time.sleep(0.1)
goal_handle = future.result()
result_future = goal_handle.get_result_async()
while not result_future.done():
    time.sleep(0.1)
print(f"LRAUV target spawned: {result_future.result().result.name}")

In [ ]:
# Spawn 30 LRAUVs in a circle of radius 50m
future = node.spawn_lrauv_circle(n_ships=30, radius_m=50)
while not future.done():
    time.sleep(0.1)
goal_handle = future.result()
result_future = goal_handle.get_result_async()
while not result_future.done():
    time.sleep(0.1)
print("30 LRAUV ships spawned around commando")

In [ ]:
stop_executor()